# SC-Mamba: Phase 1 Training Pipeline

This notebook provides the environment setup and execution commands required to train the SC-Mamba model from scratch on Google Colab (T4/V100/A100). The output weights will be saved directly into the `models/` directory, satisfying the requirements for zero-shot evaluation.

**Hardware Recommendation:** Minimum NVIDIA T4 GPU.

## 1. Environment Setup (PyTorch 2.4 + CUDA 12.1)
We use strict pre-compiled wheels for `causal-conv1d` and `mamba-ssm` to avoid 45+ minute ninja compilation times.

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.39.3 packaging triton

# Downloading compiled causal CUDNN wheels to prevent 45+ min ninja compile times
!wget -qO causal_conv1d.whl "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0%2Bcu122torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
!wget -qO mamba_ssm.whl "https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

!pip install causal_conv1d.whl mamba_ssm.whl
!pip install -r requirements.txt

## 2. Generating the Base Training Configuration
The training pipeline relies on a `.yaml` configuration to define the architecture boundaries.

In [ ]:
import yaml
import os

config = {
    "seed": 42,
    "debugging": False,
    "scaler": "min_max",
    "sin_pos_enc": False,
    "sin_pos_const": False,
    "sub_day": False,
    "encoding_dropout": 0.1,
    "handle_constants_model": True,
    "num_assets": 1,
    "ssm_config": {
        "bidirectional": False,
        "enc_conv": True,
        "init_dil_conv": True,
        "enc_conv_kernel": 5,
        "init_conv_kernel": 5,
        "init_conv_max_dilation": 3,
        "global_residual": False,
        "in_proj_norm": False,
        "initial_gelu_flag": True,
        "linear_seq": 15,
        "mamba2": True,
        "norm": True,
        "norm_type": "layernorm",
        "num_encoder_layers": 2,
        "d_state": 128,
        "residual": False,
        "token_embed_len": 1024,
    },
    "lr_scheduler": "cosine",
    "initial_lr": 1e-4,
    "learning_rate": 1e-6,
    "t_max": -1,
    "num_epochs": 50,
    "training_rounds": 1000,
    "validation_rounds": 50,
    "real_test_interval": 10,
    "real_test_datasets": ["weather", "traffic"],
    "multipoint": False,
    "sample_multi_pred": 0.0,
    "loss": "mse",
    "beta_kl": 0.1,
    "wandb": False,
    "continue_training": False,
    "model_prefix": "models"
}

os.makedirs("models", exist_ok=True)
with open("core/config.batch_ddp.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)
print("Training configuration written to core/config.batch_ddp.yaml")

## 3. Launching Pre-training on Synthetic Data
SC-Mamba, scaling upon the architectural blueprint of Mamba4Cast, learns universal temporal patterns through continuous generated mathematical series before testing on actual datasets. This generates the `.pth` weights.


In [ ]:
# Execute the main training loop
!python core/train.py -c core/config.batch_ddp.yaml

## 4. Verification
Run a quick LS to verify the model `.pth` footprint was successfully deposited into the active `/models` path bridging into Phase 2 evaluations.

In [ ]:
!ls -lh models/